# Forge Academy: Data Science Foundations
## Week 1, Session 2: Thinking Like a Bank's Data Team

In the last class, we used school data to practise Python basics. Today, you will apply those skills and the ideas from EduBEE Module 1 to a fictional digital bank, ForgeBank.


### Session plan

| Time | Activity |
|---|---|
| 0:00–0:10 | Check-in and questions |
| 0:10–0:40 | Data types, sources, sampling, scale and lifecycle |
| 0:40–0:50 | Debugging clinic |
| 0:50–0:55 | Break |
| 0:55–1:25 | Build the ForgeBank account simulator together |
| 1:25–1:50 | Independent practice: loan rules, transaction flags, PIN attempts and savings |
| 1:50–2:00 | Explain a solution and review the assignment |

## 1. Check-in

- Which part of the previous notebook was hardest for you?
- What from Edube Module 1 confused or surprised you?

## 2. Module 1 in a banking context

We will apply the ideas from Edube Module 1 to a small, fictional transaction dataset.

### 2.1 Quantitative and qualitative data

- **Quantitative data** represents measurable amounts, such as transaction value, balance or age.
- **Qualitative data** describes categories or attributes, such as transaction type, channel or customer feedback.

Here are eight ForgeBank transactions:

In [25]:
amounts  = [5000, 120000, 2500, 450000, 800, 35000, 9000, 1500000]
types    = ["POS", "Transfer", "Airtime", "Transfer", "Airtime", "ATM", "POS", "Transfer"]
channels = ["Card", "App", "USSD", "App", "USSD", "Card", "Card", "App"]

print("First transaction:", amounts[0], types[0], channels[0])

First transaction: 5000 POS Card


**Discuss:** Which list contains quantitative data? Which lists contain categories? What calculation makes sense for `amounts` but not for `types`?

In [28]:
print(f"This is Adam's {amounts[0]} transaction of type {types[0]} through the {channels[0]} channel.")

This is Adam's 5000 transaction of type POS through the Card channel.


In [26]:
total = 0
for a in amounts:
    total = total + a
print(f"Total value moved: ₦{total:,}")
print(f"Average transaction: ₦{total / len(amounts):,.0f}")

Total value moved: ₦2,122,300
Average transaction: ₦265,288


With qualitative data we **count** instead:

In [29]:
transfers = 0
for t in types:
    if t == "Transfer":
        transfers = transfers + 1
print("Number of transfers:", transfers)

Number of transfers: 3


### 2.2 Data sources and collection

| Source | ForgeBank example | How it is collected | Possible limitation |
|---|---|---|---|
| System logs | Card transactions and app logins | Recorded automatically | Capture only the events the system was designed to record |
| Forms and surveys | Account opening information | Entered by a customer | Missing fields or typing errors |
| Interviews | Conversations with former customers | Asked by staff | Time consuming and usually limited in number |
| External databases and APIs | Credit bureau data or exchange rates | Requested by a system | Quality and availability depend on the provider |

Poor input data can lead to misleading analysis. Inspect this example customer record:

In [30]:
name  = "  aDEBAYO  okafor "
age   = 210          # typo: should be 21
phone = "0803 123 4567"

print("Raw:", name, age, phone)

Raw:   aDEBAYO  okafor  210 0803 123 4567


In [40]:
name.split()

['a', 'd', 'a', 'm']

In [42]:
name.strip()

'a d a m'

In [41]:
"-".join(name.split())

'a-d-a-m'

In [ ]:
name = "a d a m "
res = "-".join(name.split())
print(res)

a-d-a-m


In [44]:
rep_name = name.replace(" ", "-")
print(rep_name)

-a-d-a-m


In [43]:

clean_name  = " ".join(name.split()).title()   # split() also removes the extra spaces in the middle
clean_phone = phone.replace(" ", "")

In [34]:

if age > 120:
    print(f" Age {age} is impossible. Flag this record for checking.")

print("Clean:", clean_name, clean_phone)

 Age 210 is impossible. Flag this record for checking.
Clean: Adebayo Okafor 08031234567


### 2.3 Populations, samples and bias

- **Population:** the full group we want to learn about, such as all ForgeBank customers.
- **Sample:** the smaller group we actually study.

If ForgeBank surveys only visitors to its Victoria Island branch, it may miss customers who use only the app or USSD, and people in other cities. The results may not represent all customers.

Random selection gives each person in the **available customer list** an equal chance of selection. It cannot fix a list that already excludes important groups.

In [45]:
import random

In [49]:
customers = ["Ada", "Tunde", "Chioma", "Emeka", "Zainab", "Bola", "Kemi", "Ibrahim", "Ngozi", "Femi"]

sample = random.sample(customers, 4)
print("Random sample:", sample)


Random sample: ['Chioma', 'Ngozi', 'Tunde', 'Zainab']


In [54]:
customers[5:8]

['Bola', 'Kemi', 'Ibrahim']

In [53]:
customers[3:7]

['Emeka', 'Zainab', 'Bola', 'Kemi']

In [52]:
customers[1:4:1]

['Tunde', 'Chioma', 'Emeka']

In [50]:
customers[:4]

['Ada', 'Tunde', 'Chioma', 'Emeka']

In [37]:
biased = customers[:4]
print("Biased 'sample' (always the first 4):", biased)

Biased 'sample' (always the first 4): ['Ada', 'Tunde', 'Chioma', 'Emeka']


**Discuss:** Run the sampling cell a few times. How does random selection differ from always taking the first four names? Who would still be missing if the customer list contained only branch visitors?

### 2.4 Big data: five useful questions

| Dimension | Question | ForgeBank example |
|---|---|---|
| Volume | How much data is there? | Millions of transactions per day |
| Velocity | How quickly does it arrive? | Card transactions arrive continuously |
| Variety | What forms does it take? | Numbers, complaints, call recordings and ID photos |
| Veracity | How reliable is it? | Typos, duplicates and incomplete records |
| Value | What decision can it improve? | Reviewing suspicious activity or understanding customer needs |

**Storage terms:** A data lake can hold data in many forms; a data warehouse organises data for analysis; a data mart is a focused collection for a team or purpose. These are common patterns, not mandatory stages for every dataset.

In [55]:
transactions_per_second = 2000
per_day = transactions_per_second * 60 * 60 * 24
print(f"Transactions per day: {per_day:,}")

Transactions per day: 172,800,000


**Discuss:** A high arrival rate creates a large number of records over time. Could a person inspect every transaction in this example? What would an automated first check need to do?

### 2.5 A transaction's data lifecycle

**Creation → Storage → Processing → Analysis → Reporting → Archiving or deletion**

1. **Created:** a customer sends ₦20,000 in the app.
2. **Stored:** the transaction is recorded in a database.
3. **Processed:** the bank checks and links the transaction to the account.
4. **Analysed:** the amount contributes to a monthly summary.
5. **Reported:** a dashboard shows the summary to authorised staff.
6. **Retained or deleted:** the record is kept or removed according to applicable requirements.

Throughout this process, the bank must protect the data, control access and check quality.

### 2.6 Analysis, analytics and data science

- **Analysis:** calculate how much customers transferred last month.
- **Analytics:** investigate why USSD usage changed and communicate the finding.
- **Data science:** may combine analysis, statistics and models to address a question such as which customers may need extra support.

These practices overlap. The question and method matter more than the label.

## 3. Debugging clinic

Each of the next four cells contains one intentional error. Run a cell, read the last line of the error message, find the line responsible and repair it. Explain the fix to a partner.

In [ ]:
balance = 50000
if balance > 10000
    print("Healthy balance")

In [ ]:
deposit = "5000"      # this came from input(), so it is TEXT
new_balance = 50000 + deposit
print(new_balance)

In [ ]:
amounts = [5000, 120000, 2500]
for a in amounts:
print(a)

In [ ]:
pin = 1234
if pin = 1234:
    print("Access granted")

## 4. Build together: ForgeBank account simulator

We will build this in three steps. Type and run each step before moving on.

### Step 1: Account details

In [ ]:
owner = "Adaeze Nwosu"
balance = 25000
print(f"Account owner: {owner}")
print(f"Opening balance: ₦{balance:,}")

### Step 2: deposit and withdraw, with a rule

In [ ]:
deposit = 10000
balance = balance + deposit
print(f"Deposited ₦{deposit:,}. Balance: ₦{balance:,}")

withdrawal = 50000
if withdrawal <= balance:
    balance = balance - withdrawal
    print(f"Withdrew ₦{withdrawal:,}. Balance: ₦{balance:,}")
else:
    print(f" Insufficient funds. You tried ₦{withdrawal:,} but have ₦{balance:,}")

### Step 3: process a whole day of transactions with a loop
Positive = money in, negative = money out.

In [ ]:
balance = 25000
day = [10000, -3000, -45000, 2500, -8000, -500]

for t in day:
    if t > 0:
        balance = balance + t
        print(f" Credit  ₦{t:>8,}  | Balance ₦{balance:,}")
    elif -t <= balance:
        balance = balance + t
        print(f" Debit   ₦{-t:>8,}  | Balance ₦{balance:,}")
    else:
        print(f" Declined ₦{-t:>7,}  | Insufficient funds")

print(f"\nClosing balance: ₦{balance:,}")

**Discuss:** Which transaction was declined, and why? What information should a real bank communicate to the customer after a declined transaction?

## 5. Independent practice

Try each task yourself. If you get stuck, read the error message and explain what you expected the code to do before asking for help.

### Task A: Loan eligibility rule

In this simplified exercise, a customer qualifies if they:

- are at least 18 years old;
- earn at least ₦150,000 per month; and
- have no existing loan.

Print `"Approved"` or a specific reason for rejection. This is a coding exercise, not a real lending decision.

In [ ]:
age = 24
income = 180000
has_loan = False

# Your code here


### Task B: Transaction review flags

Flag a transaction for **review** if its amount is over ₦1,000,000 or it occurred from midnight up to, but not including, 4:00 a.m. A flag does not establish fraud.

In [ ]:
amounts = [5000, 1200000, 30000, 750000, 2500000]
hours   = [14,   10,      2,     23,     3]        # hour of day (0-23)

# Your code here: loop through both lists (hint: zip)


### Task C: PIN attempts (`while` loop)

The correct PIN is `"4321"`. The customer has at most three attempts. Use the provided list to simulate each entry. Stop once the correct PIN is entered; otherwise, report that the card is locked after three wrong attempts.

In [ ]:
correct_pin = "4321"
attempts = ["1111", "1234", "4321"]   # try changing these!

# Your code here


### Task D: Savings growth

Ada deposits ₦100,000 at a hypothetical annual interest rate of 12%. Use a loop to print the balance at the end of each year for five years. Assume interest is added once per year and there are no fees or extra deposits.

In [ ]:
balance = 100000
rate = 0.12

# Your code here


## 6. Explain your reasoning

Choose one task you solved. Explain what your code checks, what result it produces and one assumption it makes, using plain language.

## Assignment: ForgeBank Statement Analyzer

**Due:** before the Week 2 class. Submit this notebook through the course form together with your Week 1, Session 1 notebook. Complete the assigned EduBEE Module 1 and Module 2 work as well.

You are given one fictional customer's statement for the month:

In [ ]:
customer = "  tUNDE   bakare "
opening_balance = 80000
transactions = [150000, -12000, -2500, -45000, -800, 20000, -95000, -1500, -60000, 5000]
descriptions = ["salary", "pos", "airtime", "rent", "airtime", "transfer in", "transfer out", "airtime", "transfer out", "refund"]

**Task 1:** Clean the customer's name and print a statement header like `Statement for: Tunde Bakare`.

In [ ]:
# Task 1


**Task 2:** Loop through the transactions and print each one as `CREDIT` or `DEBIT`, with its description in capital letters.

In [ ]:
# Task 2


**Task 3:** Calculate total money in, total money out, and the closing balance.

In [ ]:
# Task 3


**Task 4:** Count how many transactions were **airtime** purchases and how much was spent on airtime in total.

In [ ]:
# Task 4


**Task 5:** Find the **largest single debit** using a loop (don't use `min()` or `max()`).

In [ ]:
# Task 5


**Task 6:** Assign a label based on the closing balance:

- above ₦50,000: `"Healthy"`
- ₦10,000 through ₦50,000: `"Watch spending"`
- below ₦10,000: `"Low balance alert"`

These are labels for this exercise, not financial advice.

In [ ]:
# Task 6


**Task 7 (written response):** Answer below in plain language.

1. Which statement values are quantitative, and which are qualitative?
2. Give one way the data might become inaccurate during collection.
3. The transaction data has passed through several lifecycle stages. Which stage are you working in as you calculate and explain this statement?

*Your answers:*